# Use case 4 — Resolve permitted terms via NCI EVS

**Question.** Given a USDM property with a structured codelist binding,
what are the permitted Code values?

The walk:

```
usdm:StudyDesign-studyType
  → usdm:boundCodelist ncit:C99077
  → NCI EVS REST API   /api/v1/subset/ncit/C99077/members
  → permitted terms table
```

Cases 1–3 cover the v0.1 binding layer in pure RDF. This notebook closes
the loop by calling **NCI EVS** to resolve the actual permitted Code
values for a given codelist binding.

**Why NCI EVS rather than CDISC Library REST?** CDISC contributes its
codelists to NCIt as subsets. The codelist C-Codes carried by
`usdm:boundCodelist` (`C99077`, `C188724`, `C174222`, …) *are* NCIt
subset concept IDs. NCI EVS exposes those subsets and their members
publicly — no API key, no membership tier — so this notebook runs for
anyone with internet access. CDISC Library REST stays the right choice
for full CDISC-side metadata (extensibility flags, CDISC submission
values as first-class fields, etc.) when you have member-tier access.

**Pinned to v0.1.**


## Configuration

Demo property: `StudyDesign.studyType` → `usdm:boundCodelist ncit:C99077`
(SDTM CT *Study Type*).

NCI EVS API: `https://api-evsrest.nci.nih.gov/api/v1/`

All NCIt subsets — regardless of which CDISC package contributes them —
live at the same endpoint shape. No package routing required.


In [1]:
import rdflib
import pandas as pd
import requests
from pathlib import Path

TTL_PATH = "../usdm_v4.ttl"
EXPECTED_BASELINE = 8173

if not Path(TTL_PATH).exists():
    raise FileNotFoundError(
        f"{TTL_PATH} missing — run notebooks/10/20 first to regenerate."
    )

g = rdflib.Graph()
g.parse(TTL_PATH, format="turtle")
n_triples = len(g)
print(f"parsed {n_triples} triples from {TTL_PATH}")
if n_triples != EXPECTED_BASELINE:
    print(f"  note: triple count differs from v0.1 baseline ({EXPECTED_BASELINE})"
          " — likely a DDF-RA tag bump; this notebook is pinned to v0.1 binding shape")

NCI_EVS_API_BASE = "https://api-evsrest.nci.nih.gov/api/v1"

def short(uri):
    """Last path segment, fragment-aware."""
    if uri is None:
        return None
    s = str(uri)
    if "#" in s:
        return s.split("#")[-1]
    return s.split("/")[-1]


parsed 8173 triples from ../usdm_v4.ttl


## Step 1 — find the binding for the demo property

SPARQL retrieves `usdm:boundCodelist` and `usdm:boundCodelistNote` for
the demo property. The C-code is what NCI EVS needs; the note text is
informational (tells us which CDISC package contributed the codelist).


In [2]:
DEMO_CLASS = "StudyDesign"
DEMO_ATTR  = "studyType"

BINDING_SPARQL = """
PREFIX usdm: <https://w3id.org/cdisc/usdm/v4#>

SELECT ?boundCodelist ?boundCodelistNote
WHERE {
  usdm:%s-%s usdm:boundCodelist ?boundCodelist ;
             usdm:boundCodelistNote ?boundCodelistNote .
}
""" % (DEMO_CLASS, DEMO_ATTR)

results = list(g.query(BINDING_SPARQL))
if not results:
    raise LookupError(f"no usdm:boundCodelist found for {DEMO_CLASS}.{DEMO_ATTR}")
ccode = short(results[0]["boundCodelist"])
note  = str(results[0]["boundCodelistNote"])
print(f"property:  usdm:{DEMO_CLASS}-{DEMO_ATTR}")
print(f"codelist:  ncit:{ccode}")
print(f"note:      {note}")


property:  usdm:StudyDesign-studyType
codelist:  ncit:C99077
note:      Y (SDTM Terminology Codelist C99077)


## Step 2 — call NCI EVS

Endpoint: `{base}/subset/ncit/{ccode}/members`. Open API, no
authentication required. The `include=synonyms,definitions` parameter
asks for the rich payload — without it, only `code` and `name` come
back. `pageSize=1000` covers any CDISC codelist in a single call.


In [3]:
url = f"{NCI_EVS_API_BASE}/subset/ncit/{ccode}/members"
params = {"include": "synonyms,definitions", "pageSize": 1000}

print(f"GET {url}")
response = requests.get(url, params=params, timeout=30)
print(f"HTTP {response.status_code}")
if response.status_code != 200:
    print(response.text[:500])
    raise RuntimeError(f"unexpected status {response.status_code}")

members = response.json()
print(f"response type:    {type(members).__name__}")
print(f"members returned: {len(members)}")
print(f"first member:     {members[0]['code']} — {members[0]['name']}")


GET https://api-evsrest.nci.nih.gov/api/v1/subset/ncit/C99077/members
HTTP 200
response type:    list
members returned: 4
first member:     C98722 — Expanded Access Study


HTTP 200
response type:    list
members returned: 4
first member:     C98722 — Expanded Access Study


## Step 3 — extract permitted terms

Each member is an NCIt concept with `code`, `name`, `synonyms`, and
`definitions`. The CDISC submission value lives in `synonyms` as the
entry where `source="CDISC"` and `termType="PT"` — the all-caps form
CDISC uses on case report forms. The NCI preferred term is the
concept's `name`. Definitions come from multiple sources (NCI, CDISC,
FDA-NIH, …); we surface the CDISC one when present, else NCI.


In [4]:
def cdisc_submission_value(member):
    """Extract the CDISC submission value from a member's synonyms (or None)."""
    for s in member.get("synonyms", []) or []:
        if s.get("source") == "CDISC" and s.get("termType") == "PT":
            return s.get("name")
    return None

def best_definition(member):
    """Prefer CDISC definition; fall back to NCI; else first available."""
    defns = member.get("definitions", []) or []
    for src in ("CDISC", "NCI"):
        for d in defns:
            if d.get("source") == src:
                return d.get("definition")
    return defns[0].get("definition") if defns else None

def members_to_df(members):
    rows = []
    for m in members:
        rows.append({
            "conceptId":       m.get("code"),
            "submissionValue": cdisc_submission_value(m),
            "preferredTerm":   m.get("name"),
            "definition":      best_definition(m),
        })
    return pd.DataFrame(rows)

terms_df = members_to_df(members)
print(f"=== {ccode} permitted terms ({len(terms_df)} rows) ===")
terms_df


=== C99077 permitted terms (4 rows) ===


,conceptId,submissionValue,preferredTerm,definition
0,C98722,EXPANDED ACCESS,Expanded Access Study,Studies that provide a means for obtaining an ...
1,C98388,INTERVENTIONAL,Interventional Study,Studies in which individuals are assigned by a...
2,C16084,OBSERVATIONAL,Observational Study,Study in which the researchers observe the eff...
3,C129000,PATIENT REGISTRY,Patient Registry Study,Observational studies which include an organiz...


## Step 4 — refactor into a reusable function

`permitted_terms_for(class_name, attr_name)` runs the full pipeline:
SPARQL for the binding → NCI EVS call → terms DataFrame.
Returns `(ccode, terms_df)`. Raises if the property has no structured
binding.


In [5]:
def permitted_terms_for(class_name, attr_name):
    """Resolve permitted terms for a USDM property's bound codelist via NCI EVS."""
    sparql = """
    PREFIX usdm: <https://w3id.org/cdisc/usdm/v4#>
    SELECT ?boundCodelist
    WHERE {
      usdm:%s-%s usdm:boundCodelist ?boundCodelist .
    }
    """ % (class_name, attr_name)
    results = list(g.query(sparql))
    if not results:
        raise LookupError(f"no usdm:boundCodelist for usdm:{class_name}-{attr_name}")
    cc = short(results[0]["boundCodelist"])
    url = f"{NCI_EVS_API_BASE}/subset/ncit/{cc}/members"
    response = requests.get(url, params={"include": "synonyms,definitions", "pageSize": 1000}, timeout=30)
    if response.status_code != 200:
        raise RuntimeError(f"HTTP {response.status_code} for {url}: {response.text[:200]}")
    return cc, members_to_df(response.json())


In [ ]:
# Demo on two more properties — different source categories from cases 1–3
for cls, attr in [("Organization", "type"), ("StudyArm", "type")]:
    print(f"--- usdm:{cls}-{attr} ---")
    cc, df = permitted_terms_for(cls, attr)
    print(f"  codelist: ncit:{cc} ({len(df)} terms)")
    print(df.to_string(index=False))
    print()


--- usdm:Organization-type ---


  codelist: ncit:C174222 (7 terms)
conceptId        submissionValue          preferredTerm                                                                                                                                                                                                                       definition
  C174267  Active Comparator Arm  Active Comparator Arm                                                                                                                                                                                         An arm describing the active comparator.
  C174226            Control Arm            Control Arm An arm describing the intervention or treatment plan for a group of participants in the study receiving a control. The control may comprise a non-investigational product (active control) or regimen, placebo, or no treatment.
  C174266       Experimental Arm    Investigational Arm                                                                   

## Discussion

What this enables:

The v0.1 binding layer plus one NCI EVS call closes the loop from
*"what does USDM say about this property?"* to *"here are the actual
permitted Code values."* With the function from step 4, any of the 45
structured bindings is one call away — and there's no API key.

Why this works:

CDISC contributes its terminology to NCIt as subsets. The codelist
C-Codes in `usdm:boundCodelist` *are* NCIt subset IDs. Every `Y
(SDTM Terminology Codelist Cxxxxxx)` and `Y (Protocol Terminology
Codelist Cxxxxxx)` and bare `Y (Cxxxxxx)` resolves through the same
NCI EVS endpoint shape, regardless of source category — NCI EVS is the
common upstream.

Limitations:

- **Free-text bindings.** The 12 free-text bindings (ISO 3166, ISO 639,
  MedDRA, SNOMEDCT, FHIR, etc.) point to external dictionaries with no
  single REST API; they need human interpretation case-by-case.
- **CDISC-specific metadata fields.** NCI EVS gives the canonical NCIt
  view; some CDISC-side fields (`extensible` flag, distinct CDISC
  submissionValue handling beyond the synonym we extract here) are
  cleaner from CDISC Library REST — which requires CDISC member-tier
  API access. For "what are the permitted Code values?" both sources
  return the same set; for production CDISC tooling, Library REST stays
  the right choice.
- **API dependency.** Reading USDM-RDF + NCI EVS at query time keeps
  the binding loose. A tighter alternative is alignment: replace
  `usdm:boundCodelist <ncit:Cxxx>` with a Library Administered Item
  IRI, then load the relevant RDF locally and answer the same question
  via federated SPARQL with no network call. That's a future scope
  opening — deliberately not v0.1.
